# Semita Career Path Engine - local test

This notebook runs an end-to-end test of the `CareerPathEngine`: it clones the repository, installs the dependencies, and runs an example that combines the ESCO REST API with a local/open-source Hugging Face model.

The goal is to verify that, given a structured user profile, the system produces a validated `CareerPathOutput` with a roadmap, gaps, recommended resources, and interview preparation topics.


## 1. Clone the repository

This cell downloads the project from GitHub into the Colab environment. If the folder already exists, you can skip this cell or delete the directory before running it again.


In [2]:
!git clone https://github.com/Martons00/HackRome_Semita.git

Cloning into 'HackRome_Semita'...
remote: Enumerating objects: 41, done.
remote: Counting objects: 100% (41/41), done.
remote: Compressing objects: 100% (31/31), done.
remote: Total 41 (delta 11), reused 39 (delta 9), pack-reused 0 (from 0)
Receiving objects: 100% (41/41), 3.23 MiB | 7.61 MiB/s, done.
Resolving deltas: 100% (11/11), done.


## 2. Update the code and install dependencies

This cell enters the project folder, updates the repository with `git pull`, and installs the required Python libraries.

The main dependencies are:
- `transformers`, `torch`, and `accelerate` for the Hugging Face model;
- `requests` for calling the ESCO REST API;
- `pydantic` for validating structured input and output.


In [5]:
!cd HackRome_Semita && git pull && pip install -r requirements.txt

remote: Enumerating objects: 28, done.
remote: Counting objects: 100% (28/28), done.
remote: Compressing objects: 100% (11/11), done.
remote: Total 17 (delta 6), reused 17 (delta 6), pack-reused 0 (from 0)
Unpacking objects: 100% (17/17), 10.35 KiB | 2.59 MiB/s, done.
From https://github.com/Martons00/HackRome_Semita
   e45e140..d4c089b  main       -> origin/main
Updating e45e140..d4c089b
Fast-forward
 README.md                                          |   6 ++++
 career_path_engine/__init__.py                     |  30 ++++++++++++++++--
 .../__pycache__/__init__.cpython-311.pyc           | Bin 658 -> 1222 bytes
 .../__pycache__/chain.cpython-311.pyc              | Bin 4265 -> 2593 bytes
 .../__pycache__/local_hf.cpython-311.pyc           | Bin 6929 -> 6931 bytes
 career_path_engine/chain.py                        |  35 ++-------------------
 career_path_engine/local_hf.py                     |   6 ++--
 career_path_engine/retrieval.py                    |  34 ++++++++++++++++++++
 ..

## 3. Run the CareerPathEngine

This cell runs the example script `run_local_hf_career_path_engine.py`.

The executed flow is:
1. build a test `UserProfile`;
2. retrieve ESCO occupations and skills related to the target role;
3. generate the answer locally with `Qwen/Qwen2.5-1.5B-Instruct`;
4. validate the output against the `CareerPathOutput` schema.

On the first run, the model is downloaded from Hugging Face, so this step may take several minutes.


In [6]:
!python HackRome_Semita/examples/run_local_hf_career_path_engine.py

config.json: 100% 660/660 [00:00<00:00, 3.19MB/s]
tokenizer_config.json: 100% 7.30k/7.30k [00:00<00:00, 26.4MB/s]
vocab.json: 100% 2.78M/2.78M [00:00<00:00, 70.4MB/s]
merges.txt: 100% 1.67M/1.67M [00:00<00:00, 118MB/s]
tokenizer.json: 100% 7.03M/7.03M [00:00<00:00, 134MB/s]
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
model.safetensors: 100% 3.09G/3.09G [00:21<00:00, 143MB/s]
Loading weights: 100% 338/338 [00:07<00:00, 46.05it/s]
generation_config.json: 100% 242/242 [00:00<00:00, 1.73MB/s]
{
  "primary_role": "Junior Computer Vision Engineer",
  "role_match_score": 0.95,
  "fit_breakdown": {
    "fit_psychometric": 0.95,
    "fit_skills": 0.95,
    "fit_market": 0.95,
    "fit_constraints": 0.95
  },
  "gaps": [],
  "roadmap_3m": [
    "Learn more about Healthcare AI applications and trends.",
    "Build a basic resume and cover letter template.",
    "Start learning Python libraries like OpenCV and TensorFlow."
  ],
  "roadmap_6m": [
    "Expand your CV with additi

## Expected output

The final output should be a JSON object with fields such as:
- `primary_role`
- `role_match_score`
- `fit_breakdown`
- `gaps`
- `roadmap_3m`, `roadmap_6m`, `roadmap_12m`
- `recommended_resources`
- `interview_prep_topics`

If the output is not valid JSON, the engine raises an error instead of returning incomplete data. This is intentional: it makes integration with a UI or a downstream pipeline safer.


## Practical notes

To use a slightly stronger model, you can set the `HF_MODEL_ID` environment variable before the execution cell, for example `Qwen/Qwen2.5-3B-Instruct`. It will be slower and use more memory.

To change the language of ESCO results, you can set `ESCO_LANGUAGE=it`, although English often provides more stable coverage for technical roles.
